# Noisy Label Evaluation — Step-by-Step

Visualises model-assisted detection of annotation errors on a single image.

| Step | What is shown |
|------|---------------|
| 1 | All GT boxes + all predictions ≥ 0.1 (alpha ∝ confidence) |
| 2 | True Positives only (IoU ≥ 0.5, green) |
| 3 | Ignored FPs: high-confidence predictions without GT match → likely **missing labels** |
| 4 | Ignored FNs: GT boxes the model is blind to → likely **ghost / wrong labels** |
| 5 | Console output of filtered boxes + Adjusted Precision / Recall |

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import torch

# ── Configuration ─────────────────────────────────────────────────────────────
IMAGE_NAME        = "yt024_00_00046.jpg"
RUNS_DIR          = Path("../runs/rfdetr_v4")      # directory with checkpoint_best*.pth
DATA_ROOT         = Path("../Data/lars_processed") # must contain test/_annotations.coco.json
SPLIT             = "test"
IOU_THRESH        = 0.5   # IoU ≥ this → TP match
VIZ_THRESHOLD     = 0.1   # Step 1: show predictions with prob ≥ this
FP_CONF_THRESHOLD = 0.7   # Step 3: unmatched pred with prob > this → missing label
FN_CONF_THRESHOLD = 0.2   # Step 4: unmatched GT where max nearby prob < this → ghost label

# Colour palette (RGB, 0–1)
C_GT       = (0.95, 0.20, 0.20)   # red — GT boxes
C_PRED     = (1.00, 0.65, 0.00)   # orange — raw predictions (Step 1)
C_TP_PRED  = (0.13, 0.80, 0.13)   # green — TP predictions (Step 2)
C_TP_GT    = (0.60, 1.00, 0.60)   # light green — TP matched GT (Step 2)
C_IGN_FP   = (1.00, 0.50, 0.00)   # amber — ignored FPs / missing labels (Step 3)
C_IGN_FN   = (0.70, 0.15, 0.90)   # purple — ignored FNs / ghost labels (Step 4)

In [ ]:
# ── Load GT annotations & image ───────────────────────────────────────────────
ann_path = DATA_ROOT / SPLIT / "_annotations.coco.json"
ann_coco = json.load(open(ann_path))

categories  = ann_coco["categories"]
CLASS_NAMES = [c["name"] for c in categories]
CLASS_IDS   = [c["id"]   for c in categories]
id_to_name  = {c["id"]: c["name"] for c in categories}
id_to_idx   = {cid: i for i, cid in enumerate(CLASS_IDS)}
N_CLASSES   = len(CLASS_IDS)

target = next((img for img in ann_coco["images"] if img["file_name"] == IMAGE_NAME), None)
if target is None:
    raise ValueError(f"'{IMAGE_NAME}' not found in {ann_path}")

gt_anns = [a for a in ann_coco["annotations"] if a["image_id"] == target["id"]]
gt_boxes = np.array([
    [a["bbox"][0], a["bbox"][1],
     a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]]
    for a in gt_anns], dtype=float) if gt_anns else np.empty((0, 4))
gt_cats = np.array([a["category_id"] for a in gt_anns], dtype=int)

pil_img = Image.open(DATA_ROOT / SPLIT / "images" / IMAGE_NAME).convert("RGB")
img_arr = np.array(pil_img)

print(f"Image : {IMAGE_NAME}  ({img_arr.shape[1]}×{img_arr.shape[0]})")
print(f"GT boxes: {len(gt_boxes)}")
for cid, name in id_to_name.items():
    n = int((gt_cats == cid).sum())
    if n:
        print(f"  [{cid:2d}] {name}: {n}")

In [ ]:
# ── Load finetuned model & run inference at threshold=0.0 ─────────────────────
# Requires a finetuned checkpoint in RUNS_DIR.
# Train with: python ../4_Model/train_rfdetr.py
from rfdetr import RFDETRBase

ckpt_candidates = sorted(RUNS_DIR.glob("checkpoint_best*.pth")) if RUNS_DIR.exists() else []
if not ckpt_candidates:
    raise FileNotFoundError(
        f"No finetuned checkpoint found in {RUNS_DIR}.\n"
        f"Run '../4_Model/train_rfdetr.py' first, or point RUNS_DIR at an existing run."
    )
best_ckpt = ckpt_candidates[-1]

model = RFDETRBase(resolution=560, num_classes=N_CLASSES, pretrain_weights=None)
state = torch.load(best_ckpt, map_location="cpu")
model.model.model.load_state_dict(state["model"])
print(f"Loaded : {best_ckpt.name}")

# threshold=0.0 → all raw detections, needed for Steps 3 & 4
raw_preds   = model.predict(pil_img, threshold=0.0)
pred_boxes  = raw_preds.xyxy        if len(raw_preds) > 0 else np.empty((0, 4))
pred_scores = (raw_preds.confidence
               if raw_preds.confidence is not None and len(raw_preds) > 0
               else np.ones(len(raw_preds)))
pred_cids   = raw_preds.class_id    if len(raw_preds) > 0 else np.array([], dtype=int)
print(f"Raw predictions (threshold=0.0): {len(pred_boxes)}")

In [ ]:
# ── IoU helper + greedy matching ──────────────────────────────────────────────
def iou(a, b):
    xi1, yi1 = max(a[0],b[0]), max(a[1],b[1])
    xi2, yi2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0., xi2-xi1) * max(0., yi2-yi1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/union if union > 0 else 0.


def max_conf_within(gt_box):
    """Max confidence of any prediction with any spatial overlap (IoU > 0) with gt_box."""
    scores = [pred_scores[i] for i in range(len(pred_boxes))
              if iou(pred_boxes[i], gt_box) > 0]
    return max(scores) if scores else 0.0


# Greedy matching: highest-confidence predictions get first pick
order = np.argsort(-pred_scores) if len(pred_scores) > 0 else np.array([], dtype=int)
pb = pred_boxes[order]   # sorted pred boxes
ps = pred_scores[order]  # sorted scores
pc = pred_cids[order]    # sorted class ids

matched_gt, matched_pred = {}, {}
for pi in range(len(pb)):
    best_iou, best_gi = IOU_THRESH, -1
    for gi in range(len(gt_boxes)):
        if gi in matched_gt:
            continue
        v = iou(pb[pi], gt_boxes[gi])
        if v > best_iou:
            best_iou, best_gi = v, gi
    if best_gi >= 0:
        matched_gt[best_gi]  = pi
        matched_pred[pi]     = best_gi

# ── Partition results ─────────────────────────────────────────────────────────
# True Positives: (gt_box, gt_cat, pred_box, pred_cat, score)
tp_pairs = [(gt_boxes[gi], gt_cats[gi], pb[pi], pc[pi], ps[pi])
            for gi, pi in matched_gt.items()]

# All unmatched predictions
fp_all = [(pb[pi], pc[pi], ps[pi]) for pi in range(len(pb)) if pi not in matched_pred]

# All unmatched GT boxes + max confidence within each
fn_with_mc = [(gt_boxes[gi], gt_cats[gi], max_conf_within(gt_boxes[gi]))
              for gi in range(len(gt_boxes)) if gi not in matched_gt]

# Step 3: high-conf FPs → likely missing labels
ignored_fp = [(b,c,s) for b,c,s in fp_all if s > FP_CONF_THRESHOLD]
regular_fp = [(b,c,s) for b,c,s in fp_all if s <= FP_CONF_THRESHOLD]

# Step 4: unmatched GTs where model is completely blind → likely ghost labels
ignored_fn = [(b,c,mc) for b,c,mc in fn_with_mc if mc < FN_CONF_THRESHOLD]
regular_fn = [(b,c,mc) for b,c,mc in fn_with_mc if mc >= FN_CONF_THRESHOLD]

print(f"TP                      : {len(tp_pairs)}")
print(f"FP regular              : {len(regular_fp)}")
print(f"FP ignored (conf>{FP_CONF_THRESHOLD}) : {len(ignored_fp)}  ← potential missing labels")
print(f"FN regular              : {len(regular_fn)}")
print(f"FN ignored (max<{FN_CONF_THRESHOLD}) : {len(ignored_fn)}  ← potential ghost labels")

---
## Step 1 — Basis-Analyse

All GT boxes (solid red) overlaid with **all** model predictions with probability ≥ 0.1.  
The prediction box opacity (alpha) is coupled directly to the predicted confidence score.

In [ ]:
def _draw(ax, box, color_rgb, alpha=1.0, lw=2, ls='-', label=None):
    """Draw one bounding box outline on axes. Alpha applies to edge and label."""
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=max(0.5, lw*alpha),
        edgecolor=(*color_rgb, alpha),
        facecolor='none', linestyle=ls
    ))
    if label is not None:
        ax.text(x1+2, max(y1-4, 4), label, fontsize=6, color='white', clip_on=True,
                bbox=dict(facecolor=color_rgb, alpha=min(1.0, alpha+0.15),
                          pad=1, boxstyle='round'))


fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img_arr)

# GT boxes — solid red
for i, (box, cid) in enumerate(zip(gt_boxes, gt_cats)):
    _draw(ax, box, C_GT, alpha=1.0, lw=2,
          label=id_to_name.get(int(cid), str(cid)))

# Predictions — alpha = confidence score
mask_viz = pred_scores >= VIZ_THRESHOLD
for box, score, cid in zip(pred_boxes[mask_viz], pred_scores[mask_viz], pred_cids[mask_viz]):
    label = f"{score:.2f}" if score >= 0.5 else None   # only annotate higher-conf ones
    _draw(ax, box, C_PRED, alpha=float(score), lw=2.5, label=label)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], color=C_GT,   lw=2,   label=f'GT ({len(gt_boxes)} boxes)'),
    Line2D([0],[0], color=C_PRED, lw=2,   label=f'Predictions ≥ {VIZ_THRESHOLD} ({mask_viz.sum()} shown)\nalpha ∝ confidence'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.9)
ax.set_title(f"Step 1 — Basis-Analyse: GT (red) + Predictions (orange, alpha ∝ prob)\n"
             f"{IMAGE_NAME}", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 2 — Korrekte Zuordnungen (True Positives)

Only predictions matched to a GT box at IoU ≥ 0.5 (green).  
Matched GT boxes are drawn with dashed light-green outlines for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img_arr)

for gt_box, gt_cid, pred_box, pred_cid, score in tp_pairs:
    # Matched GT — dashed light green
    _draw(ax, gt_box, C_TP_GT, alpha=0.7, lw=1.5, ls='--')
    # Matched prediction — solid green with score label
    _draw(ax, pred_box, C_TP_PRED, alpha=1.0, lw=2.5,
          label=f"TP {score:.2f}")

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color=C_TP_PRED, lw=2.5, label=f'TP prediction ({len(tp_pairs)})'),
    Line2D([0],[0], color=C_TP_GT,   lw=1.5, ls='--', label='Matched GT box'),
], loc='upper right', fontsize=9, framealpha=0.9)
ax.set_title(f"Step 2 — True Positives (IoU ≥ {IOU_THRESH})  ·  {len(tp_pairs)} matches\n"
             f"{IMAGE_NAME}", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 3 — Fehlende Labels (Ignored FPs)

Predictions with **confidence > 0.7** that have no matching GT box (IoU < 0.5).  
The model is very certain about these detections, but the annotator left no label — likely **missing annotations**.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img_arr)

# All GT boxes as thin reference lines
for box, cid in zip(gt_boxes, gt_cats):
    _draw(ax, box, C_GT, alpha=0.35, lw=1, ls='--')

# Ignored FPs — amber, solid
for box, cid, score in ignored_fp:
    _draw(ax, box, C_IGN_FP, alpha=1.0, lw=2.5,
          label=f"FP {score:.2f}  missing?")

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color=C_GT,     lw=1, ls='--', alpha=0.35, label='GT (reference)'),
    Line2D([0],[0], color=C_IGN_FP, lw=2.5,        label=f'Ignored FP conf>{FP_CONF_THRESHOLD} ({len(ignored_fp)}) → missing label?'),
], loc='upper right', fontsize=9, framealpha=0.9)
ax.set_title(f"Step 3 — Fehlende Labels: high-conf FPs without GT match\n"
             f"conf > {FP_CONF_THRESHOLD}  ·  {len(ignored_fp)} candidate(s)  ·  {IMAGE_NAME}",
             fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 4 — Geister-Labels (Ignored FNs)

GT boxes where the **maximum model confidence of any overlapping prediction is < 0.2**.  
The model sees nothing here even at minimum threshold — likely **ghost / incorrectly placed GT labels**.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img_arr)

# All GT boxes as thin reference lines
for box, cid in zip(gt_boxes, gt_cats):
    _draw(ax, box, C_GT, alpha=0.35, lw=1, ls='--')

# Ignored FNs — purple, solid
for box, cid, mc in ignored_fn:
    _draw(ax, box, C_IGN_FN, alpha=1.0, lw=2.5,
          label=f"{id_to_name.get(int(cid), str(cid))}  max={mc:.2f} ghost?")

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color=C_GT,     lw=1, ls='--', alpha=0.35, label='GT (reference)'),
    Line2D([0],[0], color=C_IGN_FN, lw=2.5,        label=f'Ignored FN max_conf<{FN_CONF_THRESHOLD} ({len(ignored_fn)}) → ghost label?'),
], loc='upper right', fontsize=9, framealpha=0.9)
ax.set_title(f"Step 4 — Geister-Labels: GT boxes the model is blind to\n"
             f"max nearby conf < {FN_CONF_THRESHOLD}  ·  {len(ignored_fn)} candidate(s)  ·  {IMAGE_NAME}",
             fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 5 — Metriken & Konsolen-Ausgabe

Coordinates and scores of the filtered boxes from Steps 3 and 4,  
followed by **Standard** and **Adjusted** Precision / Recall.  

- **Adjusted Precision**: ignored FPs are removed from the FP count  
- **Adjusted Recall**: ignored FNs are removed from the FN count

In [ ]:
DIVIDER = "─" * 66

# ── Step 3 boxes (missing labels) ─────────────────────────────────────────────
print(DIVIDER)
print(f"  Step 3 — Ignored FPs  (conf > {FP_CONF_THRESHOLD}, no GT match)")
print(DIVIDER)
if not ignored_fp:
    print("  none")
else:
    print(f"  {'#':>3}  {'x1':>6} {'y1':>6} {'x2':>6} {'y2':>6}  {'conf':>6}  class")
    print(f"  {'':>3}  {'':>6} {'':>6} {'':>6} {'':>6}  {'':>6}")
    for i, (box, cid, score) in enumerate(ignored_fp, 1):
        x1, y1, x2, y2 = box
        name = CLASS_NAMES[int(cid)] if 0 <= int(cid) < N_CLASSES else f"id={cid}"
        print(f"  {i:>3}  {x1:>6.1f} {y1:>6.1f} {x2:>6.1f} {y2:>6.1f}  {score:>6.4f}  {name}")

print()

# ── Step 4 boxes (ghost labels) ────────────────────────────────────────────────
print(DIVIDER)
print(f"  Step 4 — Ignored FNs  (max nearby conf < {FN_CONF_THRESHOLD}, no pred match)")
print(DIVIDER)
if not ignored_fn:
    print("  none")
else:
    print(f"  {'#':>3}  {'x1':>6} {'y1':>6} {'x2':>6} {'y2':>6}  {'max_conf':>8}  class")
    print(f"  {'':>3}  {'':>6} {'':>6} {'':>6} {'':>6}  {'':>8}")
    for i, (box, cid, mc) in enumerate(ignored_fn, 1):
        x1, y1, x2, y2 = box
        name = id_to_name.get(int(cid), f"id={cid}")
        print(f"  {i:>3}  {x1:>6.1f} {y1:>6.1f} {x2:>6.1f} {y2:>6.1f}  {mc:>8.4f}  {name}")

print()

# ── Metrics ────────────────────────────────────────────────────────────────────
tp  = len(tp_pairs)
fp  = len(fp_all)           # all unmatched predictions
fn  = len(fn_with_mc)       # all unmatched GT boxes
fp_ignored = len(ignored_fp)
fn_ignored = len(ignored_fn)
fp_adj = fp - fp_ignored    # FPs that remain after removing missing-label candidates
fn_adj = fn - fn_ignored    # FNs that remain after removing ghost-label candidates

std_prec = tp / (tp + fp)     if (tp + fp) > 0 else 0.0
std_rec  = tp / (tp + fn)     if (tp + fn) > 0 else 0.0
std_f1   = (2*std_prec*std_rec/(std_prec+std_rec)) if (std_prec+std_rec) > 0 else 0.0
adj_prec = tp / (tp + fp_adj) if (tp + fp_adj) > 0 else 0.0
adj_rec  = tp / (tp + fn_adj) if (tp + fn_adj) > 0 else 0.0
adj_f1   = (2*adj_prec*adj_rec/(adj_prec+adj_rec)) if (adj_prec+adj_rec) > 0 else 0.0

print(DIVIDER)
print(f"  Count summary")
print(DIVIDER)
print(f"  TP                            : {tp}")
print(f"  FP total                      : {fp}")
print(f"    └─ regular FP (kept)        : {fp_adj}")
print(f"    └─ ignored FP (missing GT)  : {fp_ignored}  (conf > {FP_CONF_THRESHOLD})")
print(f"  FN total                      : {fn}")
print(f"    └─ regular FN (kept)        : {fn_adj}")
print(f"    └─ ignored FN (ghost GT)    : {fn_ignored}  (max conf < {FN_CONF_THRESHOLD})")
print()
print(DIVIDER)
print(f"  {'Metric':<24} {'Standard':>10} {'Adjusted':>10}")
print(DIVIDER)
print(f"  {'Precision':<24} {std_prec:>10.4f} {adj_prec:>10.4f}")
print(f"  {'Recall':<24} {std_rec:>10.4f} {adj_rec:>10.4f}")
print(f"  {'F1':<24} {std_f1:>10.4f} {adj_f1:>10.4f}")
print(DIVIDER)